# Cleaning text & categorical values: one thing, one label

_Data Wrangling_

**Squash the many spellings of the same category down to one, before anyone counts or joins on it.**

A free-text or categorical column is a column of names:
       a country, a city, a product type, a status. The trouble is that humans (and the systems
       that log them) write the same thing many ways. The United States shows up as
       USA, usa, U.S.A.,  US , and
       United States of America &mdash; all the same country, all different strings.

       To a computer those are different values. So a count of countries reports five tiny
       groups instead of one big one. A join on country silently drops the rows whose spelling does
       not match. A model sees five rare categories where there is really one common one. The signal
       gets split and watered down.

---

This notebook collapses the many spellings down to one, a stage at a time — audit, normalize, standardize, then bucket the rare tail. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q rapidfuzz
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — pandas

### Step 1 — Audit how many spellings exist

You can't fix what you haven't seen. We start with a messy `country` column straight out of a form, and call `value_counts()` to surface every distinct string. The count of distinct labels — 38 — is the size of the problem: dozens of strings that really name only a handful of countries.

In [ ]:
import pandas as pd
import unicodedata

# A messy categorical column straight out of a form / merged sources.
df = pd.DataFrame({'country': [
    ' USA ', 'usa', 'U.S.A.', 'United States', 'US', 'U.S.',
    'United States of America', 'usa', 'USA', ' us ', 'U.S', 'Unted States',
    'UK', 'U.K.', 'United Kingdom', 'uk', 'Great Britain', 'U.K', 'United  Kingdom',
    'Germany', 'germany', 'DE', 'Deutschland', ' Germany',
    'France', 'france', 'FR', 'Fance',
    'Canada', 'CA', 'canada',
    'Brazil', 'Brasil', 'BR',
    'Japan', 'JP', 'japan',
    'India', 'Australia',
]})

# AUDIT: how many ways is the same thing written?
print(df['country'].value_counts())
print('distinct labels BEFORE:', df['country'].nunique())   # -> 38

### Step 2 — Normalize away the cosmetic noise

A lot of the variation is purely cosmetic: leading/trailing spaces, mixed case, doubled spaces, dots and hyphens, and accented letters. We strip, upper-case, squeeze whitespace, drop punctuation, and fold accents via Unicode normalization. None of this understands what the strings *mean* — it just collapses `' USA '`, `'usa'`, and `'U.S.A.'` into a single `'USA'`. That alone takes us from 38 distinct labels down to 23.

In [ ]:
# NORMALIZE: strip, upper-case, squeeze whitespace, drop punctuation.
s = (df['country']
     .str.strip()                          # kill leading/trailing spaces
     .str.upper()                          # fold case: 'usa' -> 'USA'
     .str.replace(r'\s+', ' ', regex=True) # collapse 'UNITED  KINGDOM' -> 'UNITED KINGDOM'
     .str.replace(r'[.\-]', '', regex=True))# drop dots/hyphens: 'U.S.A.' -> 'USA'

# Strip accents (e.g. 'MÉXICO' -> 'MEXICO') via Unicode normalization.
def strip_accents(x):
    decomposed = unicodedata.normalize('NFKD', x)
    return ''.join(c for c in decomposed if not unicodedata.combining(c))

s = s.map(strip_accents)
print('distinct AFTER normalize:', s.nunique())              # -> 23

### Step 3 — Standardize every variant to one canonical label

Normalizing can't know that `'USA'` and `'UNITED STATES'` are the same country, or that `'FANCE'` is a typo for France. That needs an explicit mapping from each known variant to a single canonical label. Applying it with `replace` merges the synonyms and typos, dropping us from 23 distinct labels to 9 — and now each country's count is the true sum of its fragments.

In [ ]:
# STANDARDIZE: map every variant (incl. typos) to one canonical label.
canon = {
    'USA': 'United States', 'US': 'United States', 'UNITED STATES': 'United States',
    'UNITED STATES OF AMERICA': 'United States', 'UNTED STATES': 'United States',
    'UK': 'United Kingdom', 'UNITED KINGDOM': 'United Kingdom', 'GREAT BRITAIN': 'United Kingdom',
    'GERMANY': 'Germany', 'DE': 'Germany', 'DEUTSCHLAND': 'Germany',
    'FRANCE': 'France', 'FR': 'France', 'FANCE': 'France',          # 'FANCE' is a typo
    'CANADA': 'Canada', 'CA': 'Canada',
    'BRAZIL': 'Brazil', 'BRASIL': 'Brazil', 'BR': 'Brazil',
    'JAPAN': 'Japan', 'JP': 'Japan',
    'INDIA': 'India', 'AUSTRALIA': 'Australia',
}
df['country_clean'] = s.replace(canon)
print('distinct AFTER standardize:', df['country_clean'].nunique())  # -> 9
print(df['country_clean'].value_counts())
# United States 12 | United Kingdom 7 | Germany 5 | France 4 | ...

# (optional) fuzzy-match leftover typos against the known good labels:
# from rapidfuzz import process, fuzz
# known = ['United States', 'United Kingdom', 'Germany', 'France', ...]
# process.extractOne('Unted States', known, scorer=fuzz.ratio)  # -> ('United States', 92, ...)

### Step 4 — Collapse the rare tail into 'Other'

Even after standardizing, a few categories appear only once (India, Australia). For modeling, ultra-rare categories give unreliable weights and balloon one-hot width. We compute frequencies, find the labels appearing fewer than 2 times, and fold them into a single `'Other'` bucket. Note this cleans the *values*; turning them into model columns (one-hot / dummy / effect) is the separate next step.

In [ ]:
# COLLAPSE RARE categories into 'Other' (frequency threshold).
freq = df['country_clean'].value_counts()
rare = freq[freq < 2].index                  # appears fewer than 2 times
is_rare = df['country_clean'].isin(rare)
df['country_final'] = df['country_clean'].where(~is_rare, 'Other')
print(df['country_final'].value_counts())     # rare singletons folded into 'Other'

# NOTE: cleaning the VALUES is done. ENCODING them for a model
# (one-hot / dummy / effect) is the separate next step -> fe-categorical-encoding.

## Visualize the data & results

_One country (the United States) is typed 10 different ways across 12 rows. What do the category counts look like before cleaning versus after standardizing to canonical labels?_

### Step 1 — Show the before: one country, ten spellings

To make the problem vivid we isolate the United States family of labels. The raw column has 38 distinct strings overall; within just the 12 US rows there are 10 different spellings. Counting them confirms how badly a single category is fragmented before any cleaning.

In [ ]:
import pandas as pd

raw = [
    ' USA ', 'usa', 'U.S.A.', 'United States', 'US', 'U.S.',
    'United States of America', 'usa', 'USA', ' us ', 'U.S', 'Unted States',
    'UK', 'U.K.', 'United Kingdom', 'uk', 'Great Britain', 'U.K', 'United  Kingdom',
    'Germany', 'germany', 'DE', 'Deutschland', ' Germany',
    'France', 'france', 'FR', 'Fance',
    'Canada', 'CA', 'canada',
    'Brazil', 'Brasil', 'BR',
    'Japan', 'JP', 'japan',
    'India', 'Australia',
]
s = pd.Series(raw)
print('distinct BEFORE:', s.nunique())          # -> 38

# The US family of labels, lightly stripped for display.
us_keys = {'USA', 'US', 'UNITEDSTATES', 'UNITEDSTATESOFAMERICA', 'UNTEDSTATES'}
us_family = [x.strip() for x in raw
             if x.strip().upper().replace('.', '').replace(' ', '') in us_keys]
print(pd.Series(us_family).value_counts())
# USA 2 | usa 2 | U.S.A. 1 | United States 1 | US 1 | U.S. 1 | ... 12 rows, 10 labels

### Step 2 — Show the after: counts on canonical labels

Now we normalize and map to canonical labels and recount. The 38 spellings collapse to 9 real countries, and the United States — previously scattered across 10 fragments — lands as a single row of 12, the largest group. This is the whole payoff of the cleaning pipeline in one comparison.

In [ ]:
# Clean: normalize then map to canonical labels.
norm = (s.str.strip().str.upper()
         .str.replace(r'\s+', ' ', regex=True)
         .str.replace('.', '', regex=False))
canon = {
    'USA': 'United States', 'US': 'United States', 'UNITED STATES': 'United States',
    'UNITED STATES OF AMERICA': 'United States', 'UNTED STATES': 'United States',
    'UK': 'United Kingdom', 'UNITED KINGDOM': 'United Kingdom', 'GREAT BRITAIN': 'United Kingdom',
    'GERMANY': 'Germany', 'DE': 'Germany', 'DEUTSCHLAND': 'Germany',
    'FRANCE': 'France', 'FR': 'France', 'FANCE': 'France',
    'CANADA': 'Canada', 'CA': 'Canada',
    'BRAZIL': 'Brazil', 'BRASIL': 'Brazil', 'BR': 'Brazil',
    'JAPAN': 'Japan', 'JP': 'Japan', 'INDIA': 'India', 'AUSTRALIA': 'Australia',
}
clean = norm.replace(canon)
print('distinct AFTER:', clean.nunique())        # -> 9
print(clean.value_counts())
# United States 12 | United Kingdom 7 | Germany 5 | France 4 |
# Canada 3 | Brazil 3 | Japan 3 | India 1 | Australia 1

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A "sales by country" report lists USA, usa, U.S.A., and United States as four separate rows, so the US never appears in the top 3. Walk through the cleaning steps that fix it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Audit with df['country'].value_counts() to see the near-duplicate labels. — _You can't fix what you haven't seen; the audit reveals that one country is split across several strings._
- Normalize: .str.strip().str.upper().str.replace('.', '', regex=False). — _Trimming spaces, folding case, and dropping dots collapses " USA ", "usa", and "U.S.A." into one "USA"._
- Standardize: .replace({"USA":"United States","US":"United States", ...}). — _Casing alone can't merge "USA" with "United States"; an explicit mapping to a canonical label does._
- Re-run value_counts() to confirm one US row with the summed count. — _The report now ranks countries by their true totals instead of by spelling fragments._

**Answer:** Audit to see the variants, normalize (strip + upper + drop punctuation) to kill spacing/case/dot noise, then standardize with a replace mapping to a single canonical "United States". After that the four fragments become one row whose count is their sum, and the US lands in its rightful rank.

</details>

**Problem 2.** You inner-join an orders table on country with a reference table, and ~40% of orders silently disappear from the result. The orders use values like "usa" and " UK "; the reference table uses "USA" and "UK". What is happening and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that joins match keys exactly, including case and whitespace. — _"usa" and "USA" are different strings, so those rows find no match and are dropped by an inner join with no error._
- Normalize the join key on both tables the same way: .str.strip().str.upper(). — _Making both keys identical in case and whitespace lets the matching rows actually meet._
- Then join on the normalized key. — _With consistent keys the previously-dropped rows are retained, recovering the missing 40%._

**Answer:** The join is case- and whitespace-sensitive, so "usa" never matches "USA" and those rows are dropped silently. Fix it by normalizing the join key on both sides (.str.strip().str.upper(), and ideally the same canonical mapping) before joining. This is the "case-sensitive joins failing silently" pitfall.

</details>

**Problem 3.** Your city column has 4,000 distinct values, most appearing only once or twice, and one-hot encoding it produces 4,000 columns. You decide to collapse rare cities into "Other" using a frequency threshold. What is the benefit, and what must you check before trusting the result?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute frequencies with value_counts() and pick a threshold (e.g. keep cities with count &ge; 20). — _Cities below the threshold are too rare to give a reliable model weight; bucketing them bounds the cardinality._
- Map sub-threshold cities to "Other". — _This shrinks 4,000 mostly-empty one-hot columns down to the common cities plus a single catch-all column._
- Inspect what landed in "Other" before trusting it. — _A rare-but-important city (a high-value market) could be hidden in the bucket; the "Other hiding a real category" pitfall._

**Answer:** Benefit: collapsing the rare tail into "Other" caps the cardinality, so one-hot encoding produces a handful of populated columns instead of 4,000 mostly-zero ones &mdash; less memory, faster training, more reliable weights. Check: look at what fell into "Other", because a frequency threshold can sweep a small-but-important category into the bucket and hide it.

</details>